<!-- WBPROJ_PATCHED -->
# Sandbox — gender-classification rerank (test environment)

This is the experimental rerank pipeline used while developing the India
gender-classifier. The intermediate files it reads/writes live in
`archive/india/test_*.xlsx`. Run from the repo root or from `notebooks/test/`.

Set `OPENAI_API_KEY` in `.env` (see `.env.example`).


In [ ]:
# Setup — load .env so OPENAI_API_KEY is available to OpenAI() automatically.
import os
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set — see .env.example"


In [ ]:
# --- Install dependencies (run only once) ---
!pip install openai pandas numpy tqdm

import os
import re
import json
import numpy as np
import pandas as pd
from tqdm import tqdm
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# --- File paths ---
INPUT_FILE   = "../../data/interim/india/MIH_S2_no_duplicates.xlsx"
FILTERED_FILE = "../../data/interim/india/MIH_S2_gender_filtered_rerank.xlsx"
FINAL_FILE    = "../../data/interim/india/MIH_S2_gender_classified.xlsx"

# --- OpenAI client setup ---
client = OpenAI()  # reads OPENAI_API_KEY from env via .env   # ⬅️ Replace with your valid key

In [ ]:
# --- Load Excel file ---
df = pd.read_excel(INPUT_FILE)
print(f"Loaded {len(df)} total rows")

# --- Optional cleaning function ---
def clean_comment(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+|https\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["clean_comment"] = df["comment"].apply(clean_comment)
df = df.dropna(subset=["clean_comment"]).reset_index(drop=True)
print(f"Cleaned dataset: {len(df)} usable comments")

In [ ]:
gender_keywords = [

    "caste","Dalit","Brahmin","norms","social norms","cultural norms","stigma","taboo",
    "ritual","ceremony","wedding","marriage","bride","groom","dowry","bride price",
    "dowry harassment","dowry death","arranged marriage","love marriage","child marriage",
    "inter-caste marriage","social boycott","kinship","family pressure","marriage pressure",
    "community elders","traditional values","custom","sin","purity","honor","family honor",
    "reputation","shame","respectability","public image","hypocrisy","societal judgment",
    "samaj","izzat","badnaam","rishta","shaadi","vivaah","ghar ki izzat","samajik daag",
    "samajik dabav","khandaan","parampara","reeti-riwaaz","biradari","jati","jaat",
    "ladki dekhna","rishta todna","log kya kahenge","izzat ka sawal","beizzati",
    "sharm aani chahiye","mooh kala",
    

    "gender equality","gender inequality","gender roles","patriarchy","male dominance",
    "male entitlement","wife duties","obedience","submission","purity","virginity",
    "chastity","victim-blaming","feminism","feminist","women","girls","men","boys",
    "husband","wife","single mother","breadwinner","man of the house","suffocating family",
    "no choice forced","control","supposed to","expected to","as a woman","as a man",
    "women should","women shouldn’t","men should","men shouldn’t",
    "ladki ghar sambhale","pati-parmeshwar","ghar ki laxmi","gharwali","pativrata",
    "sati-savitri","izzat bachaana","ladkiyon ko itna padhana kya zaroori hai",
    "ladkiyon ki maryada","mard ki baat","ladka ladki barabar","ghar ka mard",
    "bahu ki zimmedari","ladki ko chup rehna chahiye","ladki ki izzat ka sawal",


    "abuse","gaslighting","emotional manipulation","verbal abuse","physical violence",
    "coercion","marital rape","silence","trauma","red flags","victimhood",
    "financial control","emotional blackmail","abusive marriage","shattered dreams",
    "loveless marriage","no empathy","suffocation","disappointment","broken trust",
    "normalized abuse","endurance","wilted smile","fear of divorce","social stigma divorce",
    "maar-peet","atyachar","sharirik hinsa","pati ka zulm","ghar todna","chhupna",
    "samaj kya sochega","talaaq","pati ki marzi","roona-dhona","bardasht karna",
    "apmaan","izzat ka sawaal","haat uthana",


    "women’s voice","speaking up","breaking silence","liberation","independence",
    "resilience","strength","agency","choice","self-worth","career","entrepreneur",
    "professional growth","leadership","education","scholarship","economic empowerment",
    "financial independence","upward mobility","breaking stereotypes","fight together",
    "resistance","dignity","pride","girl boss",
    "apni awaaz uthana","khud ki pehchaan","swatantrata","ladkiyan kuch bhi kar sakti hain",
    "apne paon pe khadi","apni zindagi apne faisle","khud kaam karna",
    "ladkiya bojh nahi hain","nari shakti","apni izzat","apni marzi",

    "sexual health","reproductive rights","contraception","family planning",
    "safe sex","unplanned pregnancy","abortion rights","access to healthcare",
    "maternal health","menstrual health","menstrual hygiene","period poverty",
    "reproductive justice","SRHR","mahavari","periods","swachhta","garbh nirodh",
    "garbhpat","garbh niyantran","garbh dharan","maa ban’na","bachcha paida karna",
    "janam dena","swasthya seva",

    "Dalit woman","caste discrimination","caste oppression","caste privilege",
    "Brahmin supremacy","untouchability","caste-based violence","caste honor",
    "stigma caste","family honor linked to caste","dowry as caste marker",
    "jaat-bhed","chhoti jaat","badi jaat","jaat tod vivah","samaajik bahishkar",
    "jaati maryada","jaat ki izzat","ghar ki izzat jaat se judi","jaati-vaati",
    "quota waale","reservations waale",

    "fair skin","dark skin","body shaming","figure","cosmetic pressure",
    "fairness cream","beauty queen","glowing bride","appearance vs reality",
    "cosmetic surgery","societal expectations of beauty","market value",
    "objectification","trophy wife","gora rang","kaala rang","gori dulhan",
    "kaali ladki","sundarta ka pressure","khoobsurat dulhan","sundar bahu",
    "asli khoobsurti","rang pehle dekha jata hai","chehre ki chamak","Fair & Lovely",

    "fear","anger","helplessness","denial","sadness","courage","hope","empowerment",
    "relief","low self-esteem","heartbreak","rage","exhaustion","unshed tears",
    "glowing happiness","beaming pride","resilience","shame","guilt","anxiety",
    "longing","ambition","dignity","defiance","darr","gussa","majboori","nirasha",
    "dukh","himmat","umeed","apmaan","garv","shakti","pyaar","nafrat","aansu",
    "rona","dhokha","dard","sharm"
]
print(f"{len(gender_keywords)} keywords loaded for filtering.")

In [ ]:
import json
from tqdm import tqdm

def detect_language(comment):
    prompt = f"""
    Identify the primary language of this social media comment as one of:
    ["English", "Hinglish", "Hindi", "Other"].
    Hinglish = mix of Hindi and English words written in Roman script.
    Return JSON only, e.g.: {{"language": "Hinglish"}}

    Comment: "{comment}"
    """
    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        content = resp.choices[0].message.content.strip()
        data = json.loads(content) if content.startswith("{") else {"language": content}
        return data.get("language", "Unknown")
    except Exception:
        return "Error"

# --- Apply on comment column directly ---
df["language"] = [detect_language(c) for c in tqdm(df["comment"].astype(str).tolist())]

# --- Display counts ---
lang_counts = df["language"].value_counts()
lang_perc = (lang_counts / len(df) * 100).round(2)

print("\n📊 Language Distribution (comment column):")
print(pd.concat([lang_counts, lang_perc], axis=1, keys=["Count", "Percent"]))

In [ ]:
comments = df["clean_comment"].tolist()

BATCH = 100
embeddings = []

for i in tqdm(range(0, len(comments), BATCH)):
    batch = comments[i:i+BATCH]
    resp = client.embeddings.create(model="text-embedding-3-large", input=batch)
    batch_embs = [r.embedding for r in resp.data]
    embeddings.extend(batch_embs)

comment_embeddings = np.array(embeddings, dtype="float32")

# Query vector (combined keyword embedding)
query_text = " ".join(gender_keywords)
query_embedding = client.embeddings.create(
    model="text-embedding-3-large",
    input=[query_text]
).data[0].embedding

# Similarity & selection
sims = cosine_similarity(comment_embeddings, [query_embedding]).flatten()
df["similarity"] = sims

top_n = 1000
filtered_df = df.sort_values("similarity", ascending=False).head(top_n).reset_index(drop=True)
print(f"Retained top {top_n} gender-relevant comments.")
filtered_df.to_excel(FILTERED_FILE, index=False)

In [ ]:
themes = [
    "India Local Culture","Gender Norms & Dynamics","Gender-Based Violence",
    "Economic Empowerment","Sexual & Reproductive Health",
    "Caste / Intersectional","Body & Beauty Standards","Emotional States"
]

def classify_comment(comment):
    prompt = f"""
    Classify this comment into one or more of these themes: {themes}.
    Also assign sentiment: Positive, Negative, or Neutral.
    Return only JSON.
    Comment: "{comment}"
    """
    try:
        resp = client.chat.completions.create(
            model="gpt-5.1",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return json.dumps({"themes":[],"sentiment":"Error"})

classified_rows = []
BATCH_SIZE = 300

for start in range(0, len(filtered_df), BATCH_SIZE):
    batch = filtered_df.iloc[start:start+BATCH_SIZE].copy()
    print(f"Processing batch {start//BATCH_SIZE+1} ...")
    batch["classification"] = batch["clean_comment"].apply(classify_comment)
    classified_rows.append(batch)
    pd.concat(classified_rows).to_excel(FINAL_FILE, index=False)
    print(f"Saved progress up to row {start+len(batch)}")

final_df = pd.concat(classified_rows).reset_index(drop=True)

In [ ]:
def parse_classification(row):
    try:
        data = json.loads(row)
        return pd.Series([
            ",".join(data.get("themes", [])),
            data.get("sentiment", "")
        ])
    except:
        return pd.Series(["", ""])

final_df[["themes","sentiment"]] = final_df["classification"].apply(parse_classification)
final_df.to_excel(FINAL_FILE, index=False)

print(f"🎉 Done! Final classified dataset saved at: {FINAL_FILE}")

In [ ]:
print(final_df["sentiment"].value_counts())
print(final_df["themes"].value_counts().head(10))
final_df.head(10)

In [ ]:
print("Running embedding-based rerank...")

comments = df["comment"].tolist()

BATCH = 100
comment_embeddings = []
for i in range(0, len(comments), BATCH):
    batch = comments[i:i+BATCH]
    resp = client.embeddings.create(model="text-embedding-3-large", input=batch)
    batch_embs = [r.embedding for r in resp.data]
    comment_embeddings.extend(batch_embs)
    print(f"Processed {i+len(batch)} / {len(comments)} comments")

comment_embeddings = np.array(comment_embeddings, dtype="float32")

query_text = " ".join(gender_keywords)
query_embedding = client.embeddings.create(
    model="text-embedding-3-large",
    input=[query_text]
).data[0].embedding

print("Computing similarities...")
sims = cosine_similarity(comment_embeddings, [query_embedding]).flatten()

top_n = 1000
top_idx = np.argsort(sims)[-top_n:][::-1]
filtered_df = df.iloc[top_idx].reset_index(drop=True)

print(f"Original: {len(df)} | Retained top {top_n}: {len(filtered_df)}")
filtered_df.to_excel(FILTERED_FILE, index=False)

themes = [
    "India Local Culture","Gender Norms & Dynamics","Gender-Based Violence",
    "Economic Empowerment","Sexual & Reproductive Health",
    "Caste / Intersectional","Body & Beauty Standards","Emotional States"
]

def classify_comment(comment):
    prompt = f"""
    Classify this comment into one or more of these themes: {themes}.
    Also assign sentiment: Positive, Negative, or Neutral.
    Return JSON only.
    Comment: "{comment}"
    """
    try:
        resp = client.chat.completions.create(
            model="gpt-5.1",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        print("Classification error:", e)
        return json.dumps({"themes":[],"sentiment":"Error"})

BATCH_SIZE = 500
classified_rows = []

for start in range(0, len(filtered_df), BATCH_SIZE):
    batch = filtered_df.iloc[start:start+BATCH_SIZE].copy()
    print(f"Processing batch {start//BATCH_SIZE+1}...")
    batch["classification"] = batch["comment"].apply(classify_comment)
    classified_rows.append(batch)
    # Save progress incrementally
    pd.concat(classified_rows).to_excel(FINAL_FILE, index=False)
    print(f"Saved progress up to row {start+len(batch)}")

# Final merge
final_df = pd.concat(classified_rows).reset_index(drop=True)

def parse_classification(row):
    try:
        data = json.loads(row)
        return pd.Series([",".join(data.get("themes", [])), data.get("sentiment", "")])
    except:
        return pd.Series(["", ""])

final_df[["themes","sentiment"]] = final_df["classification"].apply(parse_classification)
final_df.to_excel(FINAL_FILE, index=False)

print(f"✅ Done! Final classified dataset saved at: {FINAL_FILE}")

In [ ]:
# Path to your final file
FINAL_FILE = "../../data/interim/india/MIH_S2_gender_classified.xlsx"

df = pd.read_excel(FINAL_FILE)
df = df.dropna(subset=["comment"]).reset_index(drop=True)

print(f"Loaded {len(df)} classified comments.")
df.head(3)

In [ ]:
# --- Install additional libraries if needed ---
!pip install seaborn matplotlib

import pandas as pd
import numpy as np
from collections import Counter
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sent_counts = Counter(df["sentiment"])
total = sum(sent_counts.values())
dist = {k: v/total for k,v in sent_counts.items()}
entropy = -sum(p*np.log2(p) for p in dist.values() if p > 0)

print("📊 Sentiment Distribution:")
for k,v in dist.items():
    print(f"  {k}: {v*100:.1f}%")

print(f"🔹 Sentiment Entropy: {entropy:.3f} bits")

In [ ]:
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

# Convert all sentiment values to lowercase strings
df["sentiment"] = df["sentiment"].astype(str).str.lower()

# Remove invalid entries like "nan" or "error"
df = df[~df["sentiment"].isin(["nan", "error", "none"])]

# Count frequencies
sent_counts = Counter(df["sentiment"])
total = sum(sent_counts.values())

# Compute proportions
dist = {k: v/total for k,v in sent_counts.items()}
entropy = -sum(p*np.log2(p) for p in dist.values() if p > 0)

# Print summary
print("📊 Sentiment Distribution:")
for k,v in dist.items():
    print(f"  {k}: {v*100:.1f}%")

print(f"🔹 Sentiment Entropy: {entropy:.3f} bits")

# Plot distribution
plt.bar(dist.keys(), dist.values(), color="skyblue")
plt.title("Sentiment Distribution")
plt.ylabel("Proportion")
plt.show()

In [ ]:
# --- Imports ---
import json, time, random
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns, matplotlib.pyplot as plt
from openai import OpenAI

# --- Initialize OpenAI client ---
# Replace the text in quotes below with your actual OpenAI key
client = OpenAI()  # reads OPENAI_API_KEY from env via .env

# --- Robust reclassification function ---
def reclassify(comment, retries=2):
    """
    Ask LLM to reclassify sentiment robustly.
    Includes JSON parsing and retry handling.
    """
    prompt = f"""
    Classify the sentiment of this comment strictly as one of:
    ["positive", "negative", "neutral"].
    Respond only with valid JSON like:
    {{"sentiment": "positive"}}.

    Comment: "{comment}"
    """
    for attempt in range(retries):
        try:
            resp = client.chat.completions.create(
                model="gpt-5.1",
                messages=[{"role": "user", "content": prompt}],
                temperature=0
            )
            text = resp.choices[0].message.content.strip()

            # Clean up formatting and try parsing
            text = text.replace("```json", "").replace("```", "").strip()
            data = json.loads(text)
            sent = str(data.get("sentiment", "")).lower()

            if sent not in ["positive", "negative", "neutral"]:
                raise ValueError(f"Invalid sentiment: {sent}")

            return sent
        except Exception as e:
            print(f"⚠️ Parse error on attempt {attempt+1}: {e}")
            time.sleep(0.5)
    return "error"

# --- Sample subset for evaluation ---
sample_size = min(100, len(df))  # adjustable
sample_df = df.sample(sample_size, random_state=42).copy()

# --- Apply classification with progress bar ---
results = []
for comment in tqdm(sample_df["comment"], desc="Reclassifying"):
    results.append(reclassify(comment))
    time.sleep(0.3)  # small delay to prevent rate-limit errors

sample_df["sentiment_judge"] = results

# --- Normalize both sets ---
y_true = sample_df["sentiment"].astype(str).str.lower()
y_pred = sample_df["sentiment_judge"].astype(str).str.lower()

# --- Filter out invalid ('error') cases ---
mask = (y_pred != "error")
y_true, y_pred = y_true[mask], y_pred[mask]

print(f"\nCompared {len(y_true)} valid judged samples out of {len(sample_df)}")

# --- Compute and display metrics ---
if len(y_true) > 0:
    print("\nAgreement Metrics (Proxy for Accuracy)\n")
    print(classification_report(y_true, y_pred, digits=3))

    cm = confusion_matrix(y_true, y_pred, labels=["positive","neutral","negative"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=["positive","neutral","negative"],
                yticklabels=["positive","neutral","negative"])
    plt.xlabel("Judge Prediction")
    plt.ylabel("Your Model")
    plt.title("LLM vs LLM-as-Judge (Sentiment Agreement)")
    plt.show()
else:
    print("❌ No valid predictions for comparison (all parsing failed).")